
# Incremental XGBoost (XGBRegressor) with Spearman-Guided Gene Ordering

This notebook reproduces the **incremental model** idea using **XGBoost Regressor**.
We split once (globally) into Train/Test with **20% test samples** (same split used for all proteins), then for **each protein** we:

1. Rank genes using the **`spearman`** DataFrame (rows = proteins, columns = genes).  
2. Add genes one-by-one following that order (optionally by absolute correlation).  
3. At each step, fit **XGBRegressor** on the **training set only** with **5-fold cross-validation** over a grid of hyperparameters. We use a small validation fraction within each CV fold for early stopping.  
4. Evaluate on the **held-out global test set**.  
5. Track metrics and stop when the MAE threshold/schedule is met, patience is exceeded, or `max_features` is reached.

**Conventions**  
- Samples are rows; genes/proteins are columns.  
- In the `spearman` DataFrame, **genes are columns** and **proteins are rows** (as provided).  
- We keep a single global train/test split of samples used for all proteins for consistency.



## Environment & Data
We expect these files to be available (adapt `data_dir` if needed):

- `transcriptomics_filtered.joblib`
- `proteomics_matched.joblib`
- `full_spearman100.joblib` (rows=proteins, cols=genes; correlations)

Python deps used: `numpy`, `pandas`, `scikit-learn`, `xgboost`, `scipy`, `joblib`, `tqdm`.


In [1]:

# Core imports
import os, sys, math, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from joblib import load, dump

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.utils.validation import check_is_fitted

from scipy.stats import pearsonr, spearmanr

import xgboost as xgb
from sklearn.model_selection import ParameterGrid

from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
np.random.seed(42)

print("Versions:")
print("  numpy:", np.__version__)
print("  pandas:", pd.__version__)
try:
    import sklearn
    print("  scikit-learn:", sklearn.__version__)
except Exception:
    pass
try:
    import xgboost
    print("  xgboost:", xgboost.__version__)
except Exception:
    pass


Versions:
  numpy: 1.24.4
  pandas: 2.3.2
  scikit-learn: 1.7.2
  xgboost: 2.1.3



## Paths & config
We set paths and basic options here.


In [2]:

# ---- Paths ----
data_dir = Path("data")  # change if your files live elsewhere

path_transcriptomics = data_dir / "transcriptomics_filtered.joblib"
path_proteomics     = data_dir / "proteomics_matched.joblib"
path_spearman       = data_dir / "full_spearman100.joblib"

# ---- Global split ----
TEST_SIZE = 0.20
RANDOM_STATE = 42

# ---- Incremental defaults ----
USE_ABS_CORR = True
MAX_FEATURES = 200
PATIENCE = 300
MAE_THRESHOLD = 0.5   # if set, schedule below is ignored
THRESHOLD_START = None
THRESHOLD_END = None
THRESHOLD_SCHEDULE = 'geometric'  # 'linear' or 'geometric'
MIN_IMPROVE_REL = 0.0
MIN_IMPROVE_ABS = 0.0

# ---- XGB ----
USE_GPU = False  # set True to use 'gpu_hist' if XGBoost is built with CUDA
N_SPLITS_CV = 5
EARLY_STOPPING_ROUNDS = 50
VALIDATION_FRACTION = 0.1  # within each CV fold
RANDOM_STATE_XGB = RANDOM_STATE



## Load data
We load the three joblib files and verify shapes/orientation.


In [3]:

assert path_transcriptomics.exists(), f"Missing {path_transcriptomics}"
assert path_proteomics.exists(), f"Missing {path_proteomics}"
assert path_spearman.exists(), f"Missing {path_spearman}"

transcriptomics = load(path_transcriptomics)
proteomics = load(path_proteomics)
spearman = load(path_spearman)

# Coerce to DataFrames if needed
if not isinstance(transcriptomics, pd.DataFrame):
    transcriptomics = pd.DataFrame(transcriptomics)
if not isinstance(proteomics, pd.DataFrame):
    proteomics = pd.DataFrame(proteomics)
if not isinstance(spearman, pd.DataFrame):
    spearman = pd.DataFrame(spearman)

print("transcriptomics shape:", transcriptomics.shape)  # (n_samples, n_genes)
print("proteomics shape:", proteomics.shape)            # (n_samples, n_proteins)
print("spearman shape:", spearman.shape)                # (n_proteins, n_genes) per user

# Basic checks
assert transcriptomics.index.equals(proteomics.index) or len(transcriptomics.index.intersection(proteomics.index))>0,        "We expect overlapping sample indices between transcriptomics and proteomics."

# Ensure spearman has proteins as rows, genes as cols
# (We do not transpose here; if your file differs, flip the next assert accordingly)
if set(spearman.index) != set(proteomics.columns) and set(spearman.columns) == set(proteomics.columns):
    # Looks transposed — flip it to proteins as rows
    print("Detected spearman transposed; transposing to (proteins x genes).")
    spearman = spearman.T

print("Spearman rows (proteins) example:", list(spearman.index[:5]))
print("Spearman cols (genes) example:", list(spearman.columns[:5]))


transcriptomics shape: (369, 22614)
proteomics shape: (369, 5113)
spearman shape: (5113, 51989)
Spearman rows (proteins) example: ['A2M', 'AAAS', 'AACS', 'AAGAB', 'AAK1']
Spearman cols (genes) example: ['A1BG', 'A1BG-AS1', 'A1CF', 'A2M', 'A2M-AS1']



## (Optional) Compute Spearman correlations across matched samples

We can reproduce the matching & `corrwith(..., method='spearman')` step to derive global descriptive stats or a subset like `sp85` if we want.


In [ ]:

# Add the 'src' directory to the system path and import the util module
src_dir = os.path.abspath(os.path.join(os.getcwd(), "src"))
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

try:
    import standardised_pipeline_utils
    HAVE_UTILS = True
    print("standardised_pipeline_utils imported.")
except Exception as e:
    HAVE_UTILS = False
    print("WARNING: could not import standardised_pipeline_utils; skipping optional recompute. ->", e)

if HAVE_UTILS:
    # Align samples & features for correlation (as per provided script)
    # Spearman correlation on 4843 features
    # Find matched samples with samples and proteins with genes
    transcriptomics_corr, proteomics_corr = standardised_pipeline_utils.match_proteins_samples(transcriptomics.T,
                                                                                           proteomics.T)
    spearman_correlations = transcriptomics_corr.corrwith(proteomics_corr,
                                                        axis=1,
                                                        method='spearman')
    # Descriptive statistics on spearman correlation
    spearman_correlations.describe()
    sp85_set = set(spearman_correlations[spearman_correlations > 0.85].index)
    sp85 = list(spearman_correlations[spearman_correlations > 0.85].index)
    sp85.sort(reverse=True)
    print(len(sp85))
    pd.Series(sp85, name="id").to_csv("data/sp85.csv", index=False)


standardised_pipeline_utils imported.
Number of common proteins:  4843
Number of common samples:  369



## Global train/test split (20% test)

We split **samples** once and reuse the same split for all proteins. This keeps evaluation comparable.


In [ ]:

# Intersect sample IDs
common_samples = transcriptomics.index.intersection(proteomics.index)
assert len(common_samples) > 0, "No overlapping samples between transcriptomics and proteomics."

# Restrict to intersection (to be safe)
transcriptomics = transcriptomics.loc[common_samples].copy()
proteomics = proteomics.loc[common_samples].copy()

# Global split (random; no stratification because we train many proteins)
train_samples, test_samples = train_test_split(
    common_samples.to_list(),
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
)

print(f"Train samples: {len(train_samples)}; Test samples: {len(test_samples)}; Total: {len(common_samples)}")


Train samples: 295; Test samples: 74; Total: 369



## Helper functions
We define metrics, ranking, data preparation, parameter grid, CV search with early stopping, and the incremental routine.


In [ ]:
# (Optionally keep these imports here if this cell runs standalone)
import math
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, train_test_split, ParameterGrid
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr
import xgboost as xgb

def _metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    def _safe_corr(fn, a, b):
        try:
            if len(a) < 2 or np.all(a == a[0]) or np.all(b == b[0]):
                return np.nan
            return fn(a, b)[0]
        except Exception:
            return np.nan

    pr = _safe_corr(pearsonr, y_true, y_pred)
    sr = _safe_corr(spearmanr, y_true, y_pred)

    return {"mae": float(mae), "rmse": float(rmse), "r2": float(r2),
            "pearson": float(pr) if pr==pr else np.nan,
            "spearman": float(sr) if sr==sr else np.nan}

def build_param_grid(use_gpu: bool):
    common = {
        "max_depth": [3, 6, 8],
        "min_child_weight": [1, 3],
        "gamma": [0.0, 0.1],
        "learning_rate": [0.05, 0.1],
        "n_estimators": [600, 1200, 2000],   # early stopping will clip
        "subsample": [0.7, 0.9],
        "colsample_bytree": [0.6, 0.8, 1.0],
        "reg_alpha": [0.0, 0.1, 0.5],
        "reg_lambda": [0.5, 1.0, 2.0],
        "max_bin": [256],                    # requires hist/gpu_hist
    }
    tree_method = "gpu_hist" if use_gpu else "hist"
    base = {
        "objective": ["reg:squarederror"],
        "tree_method": [tree_method],
        "verbosity": [0],
        "random_state": [RANDOM_STATE_XGB],
    }
    merged = {}
    merged.update(base)
    merged.update(common)
    return list(ParameterGrid(merged))

def _threshold_at(k, mae_threshold, threshold_start, threshold_end, max_features, threshold_schedule):
    if mae_threshold is not None:
        return float(mae_threshold)
    if threshold_start is None and threshold_end is None:
        return np.inf
    if threshold_schedule == 'linear':
        if threshold_start is None or threshold_end is None:
            raise ValueError("linear schedule needs start and end")
        return float(threshold_start + (threshold_end - threshold_start) * ((k-1)/max(1, (max_features-1))))
    if threshold_start is None or threshold_end is None:
        # simple geometric decay from start
        return float(threshold_start * (0.95 ** (k-1))) if threshold_start is not None else np.inf
    factor = (threshold_end/threshold_start) ** (1.0/max(1, (max_features-1)))
    return float(threshold_start * (factor ** (k-1)))

def get_ranked_genes_for_protein(spearman_df, protein, use_abs=True, only_genes=None):
    if protein not in spearman_df.index:
        raise KeyError(f"Protein '{protein}' not found in spearman DataFrame index.")
    s = spearman_df.loc[protein]  # series indexed by genes
    if only_genes is not None:
        s = s.loc[s.index.intersection(list(only_genes))]
    if use_abs:
        s = s.abs()
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    ranked_genes = list(s.sort_values(ascending=False).index)
    return ranked_genes

def _prepare_xy_for_genes(protein, genes, transcriptomics_df, proteomics_df, train_samples, test_samples, transform_y=None):
    # Select X (genes) and y (protein)
    X_train = transcriptomics_df.loc[train_samples, genes].copy()
    X_test  = transcriptomics_df.loc[test_samples, genes].copy()
    y_train = proteomics_df.loc[train_samples, protein].copy()
    y_test  = proteomics_df.loc[test_samples, protein].copy()

    # Drop rows with any NaNs to keep alignment consistent
    train_mask = (~X_train.isna().any(axis=1)) & (~y_train.isna())
    test_mask  = (~X_test.isna().any(axis=1))  & (~y_test.isna())
    X_train, y_train = X_train.loc[train_mask], y_train.loc[train_mask]
    X_test,  y_test  = X_test.loc[test_mask],  y_test.loc[test_mask]

    # Optional transform for y
    if transform_y == "log1p":
        y_train = np.log1p(y_train.clip(lower=0)) if (y_train.min() >= 0) else np.log1p(y_train - y_train.min() + 1)
        y_test  = np.log1p(y_test.clip(lower=0))  if (y_test.min()  >= 0) else np.log1p(y_test  - y_test.min()  + 1)
    elif transform_y in (None, "none"):
        pass
    else:
        raise ValueError(f"Unknown transform_y={transform_y}")

    return X_train, X_test, y_train.values, y_test.values

def _kfold_with_early_stopping(X, y, params, n_splits=5, val_fraction=0.1, es_rounds=50, random_state=42):
    """Manual CV for XGB with early stopping (per-fold). Returns mean RMSE across folds and details."""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    rmses, fold_details = [], []
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    for fold, (tr_idx, te_idx) in enumerate(kf.split(X), 1):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        # Inner split for early stopping validation
        if val_fraction > 0:
            X_tr2, X_val, y_tr2, y_val = train_test_split(
                X_tr, y_tr, test_size=val_fraction, random_state=random_state, shuffle=True
            )
        else:
            # If no validation fraction, just reuse the fold's test as eval_set (not ideal but functional)
            X_tr2, y_tr2 = X_tr, y_tr
            X_val, y_val = X_te, y_te

        # XGBoost >= 2.1.0: set eval_metric & early_stopping_rounds on constructor (not fit)
        model = xgb.XGBRegressor(
            **params,
            eval_metric="rmse",
            early_stopping_rounds=es_rounds
        )
        model.fit(X_tr2, y_tr2, eval_set=[(X_val, y_val)])

        preds = model.predict(X_te)
        rmse = math.sqrt(mean_squared_error(y_te, preds))
        rmses.append(rmse)
        fold_details.append({
            "fold": fold,
            "best_iteration_": int(getattr(model, "best_iteration", getattr(model, "best_ntree_limit", -1)) or -1),
            "rmse_val_fold": float(rmse),
        })

    return float(np.mean(rmses)), fold_details

def xgb_cv_search(X, y, param_grid, n_splits=5, val_fraction=0.1, es_rounds=50, random_state=42):
    """Grid search over params using KFold with per-fold early stopping. Scored by mean RMSE (lower is better)."""
    best_score = np.inf
    best_params = None
    search_log = []

    for params in param_grid:
        mean_rmse, fold_details = _kfold_with_early_stopping(
            X, y, params, n_splits=n_splits, val_fraction=val_fraction, es_rounds=es_rounds, random_state=random_state
        )
        search_log.append({"params": params, "mean_rmse": mean_rmse, "folds": fold_details})
        if mean_rmse < best_score:
            best_score = mean_rmse
            best_params = params

    return best_params, best_score, search_log

def xgb_incremental_for_protein(
    protein, transcriptomics_df, proteomics_df, spearman_df,
    train_samples, test_samples,
    use_abs_corr=True, transform_y='none',
    mae_threshold=0.5,
    threshold_start=None, threshold_end=None, threshold_schedule='geometric',
    max_features=200, min_improve_rel=0.0, min_improve_abs=0.0, patience=300,
    use_gpu=False, n_splits_cv=5, early_stopping_rounds=50, validation_fraction=0.1, random_state=42
):
    # Build ranked genes for this protein
    ranked = get_ranked_genes_for_protein(
        spearman_df, protein, use_abs=use_abs_corr, only_genes=transcriptomics_df.columns
    )
    if len(ranked) == 0:
        raise ValueError(f"No overlapping genes between spearman columns and transcriptomics columns for protein={protein}.")

    history, selected = [], []
    best_mae, no_improve = np.inf, 0
    final_model = None; final_metrics = None

    param_grid = build_param_grid(use_gpu=use_gpu)

    for g in ranked:
        cand = selected + [g]
        # Prepare data for current candidate set
        X_train, X_test, y_train, y_test = _prepare_xy_for_genes(
            protein, cand, transcriptomics_df, proteomics_df, train_samples, test_samples, transform_y=transform_y
        )
        if X_train.shape[0] < max(5, n_splits_cv):
            # not enough training samples
            continue

        # Hyperparameter search with early stopping (CV on train only)
        best_params, cv_best_rmse, cv_log = xgb_cv_search(
            X_train, y_train,
            param_grid=param_grid,
            n_splits=n_splits_cv,
            val_fraction=validation_fraction,
            es_rounds=early_stopping_rounds,
            random_state=random_state
        )

        # Fit final model on full train (with a small validation split just for early stopping)
        X_tr2, X_val, y_tr2, y_val = train_test_split(
            X_train, y_train, test_size=validation_fraction, random_state=random_state, shuffle=True
        )
        final = xgb.XGBRegressor(
            **best_params,
            eval_metric="rmse",
            early_stopping_rounds=early_stopping_rounds
        )
        final.fit(X_tr2, y_tr2, eval_set=[(X_val, y_val)])

        # Evaluate on global test split
        y_pred = final.predict(X_test)
        m = _metrics(y_test, y_pred)

        k = len(cand)
        thr = _threshold_at(k, mae_threshold, threshold_start, threshold_end, max_features, threshold_schedule)

        rel_ok = (m['mae'] <= best_mae * (1 - min_improve_rel)) if min_improve_rel > 0 else True
        abs_ok = (m['mae'] <= best_mae - min_improve_abs) if min_improve_abs > 0 else True
        improved = (m['mae'] < best_mae) and rel_ok and abs_ok

        if improved or best_mae == np.inf:
            selected, best_mae = cand, m['mae']
            no_improve = 0
            final_model = final
            final_metrics = m
            accepted = True
        else:
            accepted = False
            no_improve += 1

        history.append({
            "k": k, "threshold": thr, "accepted": accepted,
            "mae": m['mae'], "rmse": m['rmse'], "r2": m['r2'],
            "pearson": m['pearson'], "spearman": m['spearman'],
            "n_train": len(y_train), "n_test": len(y_test),
            "genes_candidate": tuple(cand), "genes_selected": tuple(selected),
            "cv_best_rmse": float(cv_best_rmse),
            "best_params": json.dumps(best_params),
            "best_iteration": int(getattr(final_model, "best_iteration", getattr(final_model, "best_ntree_limit", -1)) or -1),
        })

        if best_mae <= thr or len(selected) >= max_features or (patience is not None and no_improve >= patience):
            break

    history_df = pd.DataFrame(history)
    result = {
        "protein": protein,
        "history_df": history_df,
        "final_genes": list(history_df.iloc[-1]['genes_selected']) if len(history_df) else [],
        "final_metrics": final_metrics,
        "model": final_model
    }
    return result



## Run: choose proteins and launch the incremental search

We can run on **all proteins**, a small subset (for a smoke test), or something like `sp85` if defined above.


In [ ]:

# Choose target proteins
all_proteins = list(proteomics.columns)

# Example options:
# proteins_to_run = all_proteins                       # all
# proteins_to_run = all_proteins[:5]                   # small smoke test
# proteins_to_run = [p for p in all_proteins if p in sp85] if 'sp85' in globals() else all_proteins[:5]

#proteins_to_run = all_proteins[:5]  # default here: adjust as desired
proteins_to_run = sp85
print(f"Proteins to run: {len(proteins_to_run)} (showing first 5):", proteins_to_run[:5])

# Output folders
out_dir = Path("xgb_incremental_results")
(out_dir / "histories").mkdir(parents=True, exist_ok=True)
(out_dir / "models").mkdir(parents=True, exist_ok=True)

summ_rows = []
for protein in tqdm(proteins_to_run, desc="Proteins"):
    try:
        res = xgb_incremental_for_protein(
            protein,
            transcriptomics, proteomics, spearman,
            train_samples, test_samples,
            use_abs_corr=USE_ABS_CORR, transform_y="none",
            mae_threshold=MAE_THRESHOLD,
            threshold_start=THRESHOLD_START, threshold_end=THRESHOLD_END, threshold_schedule=THRESHOLD_SCHEDULE,
            max_features=MAX_FEATURES, min_improve_rel=MIN_IMPROVE_REL, min_improve_abs=MIN_IMPROVE_ABS, patience=PATIENCE,
            use_gpu=USE_GPU, n_splits_cv=N_SPLITS_CV, early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            validation_fraction=VALIDATION_FRACTION, random_state=RANDOM_STATE
        )
        hist = res["history_df"]
        hist.to_csv(out_dir / "histories" / f"{protein}_history.csv", index=False)

        # Save model + gene list (simple bundle)
        bundle = {
            "protein": res["protein"],
            "final_genes": res["final_genes"],
            "final_metrics": res["final_metrics"],
            "best_iteration": int(getattr(res["model"], "best_iteration", getattr(res["model"], "best_ntree_limit", -1)) or -1),
            "params": getattr(res["model"], "get_params", lambda: {})(),
        }
        dump({"bundle": bundle, "model": res["model"]}, out_dir / "models" / f"{protein}_xgb.joblib")

        fm = res["final_metrics"] or {"mae": np.nan, "rmse": np.nan, "r2": np.nan, "pearson": np.nan, "spearman": np.nan}
        summ_rows.append({
            "protein": protein,
            "n_genes_selected": len(res["final_genes"]),
            "mae_test": fm["mae"], "rmse_test": fm["rmse"], "r2_test": fm["r2"],
            "pearson_test": fm["pearson"], "spearman_test": fm["spearman"],
        })
    except Exception as e:
        print(f"[{protein}] ERROR:", e)

summary_df = pd.DataFrame(summ_rows).sort_values(by="mae_test", ascending=True)
summary_path = out_dir / "summary.csv"
summary_df.to_csv(summary_path, index=False)
summary_df.head(20)


Proteins to run: 38 (showing first 5): ['TJP2', 'SPATS2L', 'SLC16A3', 'SIRPA', 'SERPINH1']


Proteins:   0%|          | 0/38 [00:00<?, ?it/s]

[0]	validation_0-rmse:0.99688
[1]	validation_0-rmse:0.97088
[2]	validation_0-rmse:0.94074
[3]	validation_0-rmse:0.91744
[4]	validation_0-rmse:0.89104
[5]	validation_0-rmse:0.87054
[6]	validation_0-rmse:0.84937
[7]	validation_0-rmse:0.82928
[8]	validation_0-rmse:0.81231
[9]	validation_0-rmse:0.79549
[10]	validation_0-rmse:0.78220
[11]	validation_0-rmse:0.76851
[12]	validation_0-rmse:0.75114
[13]	validation_0-rmse:0.74031
[14]	validation_0-rmse:0.73007
[15]	validation_0-rmse:0.71927
[16]	validation_0-rmse:0.71019
[17]	validation_0-rmse:0.69864
[18]	validation_0-rmse:0.69447
[19]	validation_0-rmse:0.68561
[20]	validation_0-rmse:0.67506
[21]	validation_0-rmse:0.66771
[22]	validation_0-rmse:0.66087
[23]	validation_0-rmse:0.65261
[24]	validation_0-rmse:0.64945
[25]	validation_0-rmse:0.64679
[26]	validation_0-rmse:0.64316
[27]	validation_0-rmse:0.63676
[28]	validation_0-rmse:0.63536
[29]	validation_0-rmse:0.63100
[30]	validation_0-rmse:0.63129
[31]	validation_0-rmse:0.62963
[32]	validation_0-

Proteins:   0%|          | 0/38 [00:54<?, ?it/s]


KeyboardInterrupt: 


## Notes & tips
- If GPU is available, set `USE_GPU = True` (XGBoost must be built with CUDA; uses `tree_method='gpu_hist'`).
- To evaluate **log1p** transforms for `y`, set `transform_y="log1p"` in `xgb_incremental_for_protein`. Remember to invert-transform if you need raw-scale predictions downstream.
- We keep a **global** train/test split so every protein evaluates on the same held-out set.
- We stop incrementally by **MAE threshold**, **patience**, or **max_features** — whichever comes first.
- Artifacts are saved under `xgb_incremental_results/`:
  - `histories/{protein}_history.csv`: per-step diagnostics
  - `models/{protein}_xgb.joblib`: trained model and selected genes bundle
  - `summary.csv`: one-row-per-protein test metrics & selected gene count
